# Themenanalyse kommunaler Bürgerhinweise

Prototyp zur Analyse von Meldetexten des Mängelmelders Konstanz mit LDA und NMF.

Fehlende und leere Texte werden seperat behandelt. Kurze Meldungen bleiben erhalten, sofern nach dem Text Preprocessing mindestens ein Token mit zwei oder mehr Buchstaben verbleibt. Meldungen ohne Merkmale im Vokabular gehen nicht in das Modelltraining ein und werden als nicht zugeordnet ausgewiesen.

## Offene Punkte

- Den erneuten Vergleich der Anzahl von Topics auswerten und die Modellwahl begründen.
- Themenbezeichnungen und Beispiele anhand der aktuellen Ergebnisse prüfen.

Die Prozentanteile beziehen sich auf alle Meldungen nach dem Text Preprocessing, einschließlich der nicht zugeordneten Meldungen. Der Anteil ohne Zuordnung wird separat ausgewiesen.

Die CSV mit den Meldungen `maengelmelder_konstanz_final_9_neu.csv` wird unter `data/raw/` erwartet. Die Zellen sind von oben nach unten auszuführen.  

Die aktuelle Analyse ergänzt `leuchte → beleuchtung` und verwendet bei der NMF-Zuordnung Koeffizienten für L2-normierte Topic-Vektoren.
Die früheren Versuche sind in [02_bisherige_vergleiche.ipynb](02_bisherige_vergleiche.ipynb) 


In [ ]:
# Schritt 01 Umgebung und Arbeitsordner prüfen

from pathlib import Path
import sys
import re

import pandas as pd
import numpy as np
import sklearn
from IPython.display import display

print("Python executable:")
print(sys.executable)

print("\nPandas Version:")
print(pd.__version__)

print("\nNumPy Version:")
print(np.__version__)

print("\nScikit Learn Version:")
print(sklearn.__version__)

print("\nAktueller Arbeitsordner:")
print(Path.cwd())


In [ ]:
# Schritt 02 Konfiguration

CSV_NAME = "maengelmelder_konstanz_final_9_neu.csv"
TEXT_COLUMN = "text_raw"
ID_COLUMN = "id"

# Kurze Meldungen bleiben erhalten, sofern nach dem Preprocessing ein Token verbleibt.
MIN_WORDS_RAW = 1
MIN_TOKENS_CLEAN = 1
MIN_TOKEN_LENGTH = 2

# Parameter der Vektorisierung
MIN_DF = 5
MAX_DF = 0.90
NGRAM_RANGE = (1, 1)
TOKEN_PATTERN = r"(?u)\b[a-zäöüß][a-zäöüß]+\b"
SUBLINEAR_TF = True

# Referenzmodelle vor dem Vergleich
N_TOPICS = 10
N_TOPICS = 10
RANDOM_STATE = 42
LDA_MAX_ITER = 20
NMF_MAX_ITER = 500

# Modelle für die Ergebnisübersichten
SELECTED_TOPIC_COUNTS = {"LDA": 10, "NMF": 12}

# Die LDA-Priors bleiben für alle verglichenen Anzahlen von Topics gleich.
LDA_DOC_TOPIC_PRIOR = 0.1
LDA_TOPIC_WORD_PRIOR = 0.1

# Erst nach der inhaltlichen Prüfung der Bezeichnungen für das aktuelle Modell aktivieren
TOPIC_LABELS_REVIEWED = True

# Deutsche Stopwörter aus dem bisherigen Prototyp
GERMAN_STOPWORDS = {
    "aber", "alle", "allem", "allen", "aller", "alles", "als", "also", "am", "an",
    "andere", "anderem", "anderen", "anderer", "anderes", "auch", "auf", "aus",
    "bei", "bin", "bis", "bist", "da", "damit", "dann", "der", "den", "des",
    "dem", "die", "das", "dass", "daß", "dazu", "dein", "deine", "deinem",
    "deinen", "deiner", "deines", "denn", "dich", "dir", "du", "dies", "diese",
    "diesem", "diesen", "dieser", "dieses", "doch", "dort", "durch", "ein",
    "eine", "einem", "einen", "einer", "eines", "einige", "einigen", "einiger",
    "einiges", "einmal", "er", "ihn", "ihm", "es", "etwas", "euer", "eure",
    "eurem", "euren", "eurer", "eures", "für", "gegen", "gewesen", "hab",
    "habe", "haben", "hat", "hatte", "hatten", "hier", "hin", "hinter", "ich",
    "mich", "mir", "ihr", "ihre", "ihrem", "ihren", "ihrer", "ihres", "euch",
    "im", "in", "indem", "ins", "ist", "jede", "jedem", "jeden", "jeder",
    "jedes", "jene", "jenem", "jenen", "jener", "jenes", "jetzt", "kann",
    "kein", "keine", "keinem", "keinen", "keiner", "keines", "können", "könnte",
    "machen", "man", "manche", "manchem", "manchen", "mancher", "manches",
    "mein", "meine", "meinem", "meinen", "meiner", "meines", "mit", "muss",
    "musste", "nach", "nicht", "nichts", "noch", "nun", "nur", "ob", "oder",
    "ohne", "sehr", "sein", "seine", "seinem", "seinen", "seiner", "seines",
    "selbst", "sich", "sie", "ihnen", "sind", "so", "solche", "solchem",
    "solchen", "solcher", "solches", "soll", "sollte", "sondern", "sonst",
    "über", "um", "und", "uns", "unser", "unsere", "unserem", "unseren",
    "unserer", "unseres", "unter", "viel", "vom", "von", "vor", "während",
    "war", "waren", "warst", "was", "weg", "weil", "weiter", "welche",
    "welchem", "welchen", "welcher", "welches", "wenn", "werde", "werden",
    "wie", "wieder", "will", "wir", "wird", "wirst", "wo", "wollen", "wollte",
    "würde", "würden", "zu", "zum", "zur", "zwar", "zwischen"
}

# Zusammengeführte Ergänzungen aus den bisherigen Versuchen.
DOMAIN_STOPWORDS = {
    "abends", "abgeholt", "abgelaufen", "abgelegt", "abgeschlossen", "abgestellt",
    "abgestellte", "abgestellter", "abgestelltes", "abholung", "adresssuche", "aktuell",
    "allerdings", "allmannsdorf", "allmannshoehe", "alte", "alter", "altes", "andere",
    "anderen", "ausgefallen", "autom", "beethovenstr", "befinden", "befindet", "beiden", "beim",
    "bereich", "bereits", "beschaedigt", "beschädigt", "bild", "bilder", "bitte", "brennen",
    "brennt", "ca", "circa", "dank", "danke", "dankeschoen", "dankeschön", "defekt", "defekte",
    "defekten", "defekter", "dienstag", "direkt", "donnerstag", "dorffriedhofsweg", "dunkel",
    "ecke", "entfernen", "entsorgt", "ermittelt", "etc", "flurweg", "foto", "fotos", "freitag",
    "frieda", "fuer", "funktion", "funktioniert", "fuss", "ganze", "gegenueber", "gegenüber",
    "gehen", "geht", "gemeldet", "gestern", "guten", "haengt", "hallo", "hans",
    "hansegartenstrasse", "haus", "hausnummer", "heugaessle", "heute", "hinten", "hoehe",
    "hoehenweg", "hängt", "höhe", "immer", "jahr", "jahren", "kaputt", "kaputte", "kaputtes",
    "kaum", "koennen", "koennte", "kommen", "kommt", "komplett", "konstanz", "konstanzer",
    "kreuzlinger", "können", "könnte", "laengerem", "leider", "leuchte", "leuchten", "leuchtet",
    "liegen", "liegt", "litzelstetten", "macairestrasse", "mainaustrasse", "mal", "mannheimer",
    "max", "mehr", "mehrere", "mehreren", "mehrfach", "mfg", "mindestens", "mittags",
    "mittwoch", "moeglich", "moegliche", "moeglicherweise", "monat", "monaten", "montag",
    "morgen", "möglich", "nacht", "naehe", "nahe", "neben", "nr", "nummer", "oben", "ort",
    "peter", "richtung", "samstag", "schaden", "schadhaft", "schaeden", "schneckenburgstrasse",
    "schoen", "schon", "schwach", "schön", "seit", "seite", "siehe", "sieht", "sigrist",
    "sonnentauweg", "sonntag", "stadt", "stark", "stehen", "steht", "steinstrasse", "stelle",
    "str", "stromeyer", "tag", "tage", "tagen", "teil", "teile", "thumb", "ueber", "unten",
    "untere", "unteren", "urisberg", "vermutlich", "viele", "vielen", "vorne", "waere",
    "waeren", "weiteres", "woche", "wochen", "worden", "wuerde", "wuerden", "wurde", "wurden",
    "wäre", "wären", "zeit", "zwei",
}

STOPWORDS_FINAL = GERMAN_STOPWORDS | DOMAIN_STOPWORDS

# Bisherige Zuordnung ausgewählter Schreibweisen und Begriffe
CANONICAL_TERMS_FINAL = {
    "abfaelle": "müll",
    "abfall": "müll",
    "abfälle": "müll",
    "aeste": "äste",
    "baeume": "baum",
    "beschaedigte": "beschädigt",
    "beschaedigten": "beschädigt",
    "beschädigte": "beschädigt",
    "bäume": "baum",
    "container": "müllcontainer",
    "fahrrads": "fahrrad",
    "fahrraeder": "fahrrad",
    "fahrräder": "fahrrad",
    "fuer": "für",
    "fussgaenger": "fußgänger",
    "fussgaengerampel": "ampel",
    "fussweg": "gehweg",
    "fusswege": "gehweg",
    "fußgängerampel": "ampel",
    "fußweg": "gehweg",
    "fußwege": "gehweg",
    "gefaehrlich": "gefährlich",
    "gehwege": "gehweg",
    "grobmuell": "grobmüll",
    "gruen": "grün",
    "gullideckel": "gulli",
    "gullis": "gulli",
    "haengt": "hängt",
    "hecken": "hecke",
    "heckenwuchs": "hecke",
    "koennen": "können",
    "laeuft": "läuft",
    "lampe": "beleuchtung",
    "lampen": "beleuchtung",
    "laterne": "beleuchtung",
    "laternen": "beleuchtung",
    "leuchte": "beleuchtung",
    "moeglich": "möglich",
    "muell": "müll",
    "muelleimer": "mülleimer",
    "müllcontainer": "müll",
    "mülltonne": "müll",
    "mülltonnen": "müll",
    "radwege": "radweg",
    "schilder": "schild",
    "straeucher": "sträucher",
    "strasse": "straße",
    "strassen": "straße",
    "strassenbeleuchtung": "beleuchtung",
    "strassenlampe": "beleuchtung",
    "strassenlampen": "beleuchtung",
    "strassenlaterne": "beleuchtung",
    "strassenlaternen": "beleuchtung",
    "strassenleuchte": "beleuchtung",
    "strassenleuchten": "beleuchtung",
    "strassenseiten": "straßenseiten",
    "strassensteiten": "straßenseiten",
    "strauch": "sträucher",
    "straßenbeleuchtung": "beleuchtung",
    "straßenlaterne": "beleuchtung",
    "straßenlaternen": "beleuchtung",
    "ueber": "über",
    "ueberfuellt": "überfüllt",
    "umgestuerzt": "umgestürzt",
    "verkehrsschild": "schild",
    "verkehrsschilder": "schild",
    "verschmutzte": "verschmutzt",
    "verschmutzten": "verschmutzt",
    "verunreinigt": "verschmutzt",
    "waechst": "wächst",
    "waere": "wäre",
    "zerstoert": "zerstört",
}


In [ ]:
# Schritt 03 CSV-Datei laden

from pathlib import Path
import pandas as pd

possible_paths = [
    Path("data") / "raw" / CSV_NAME,
    Path("..") / "data" / "raw" / CSV_NAME,
]

csv_path = None

for path in possible_paths:
    print("Prüfe:", path.resolve())
    if path.exists():
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError(
        "CSV wurde nicht gefunden. Prüfe, ob die Datei im Ordner data/raw liegt."
    )

print("\nGefundene CSV:")
print(csv_path.resolve())

# Der Projektordner ergibt sich aus dem Pfad data/raw der gefundenen CSV.
PROJECT_DIR = csv_path.parent.resolve().parent.parent

df = pd.read_csv(csv_path)

print("\nShape:")
print(df.shape)

print("\nSpalten:")
print(list(df.columns))

df.head(3)


In [ ]:
# Schritt 04 Data Profiling

required_columns = [
    "id",
    "date_created",
    TEXT_COLUMN,
    "category_parent",
    "category_child",
    "responsible",
    "lat",
    "lon",
    "address_input",
    "detail_url",
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise ValueError(f"Folgende erwartete Spalten fehlen: {missing_required}")

print("Anzahl Zeilen:", len(df))

print("\nFehlende Werte je Spalte:")
print(df.isna().sum())

print("\nFehlende Werte in text_raw:")
print(df[TEXT_COLUMN].isna().sum())

print("\nDubletten in text_raw:")
print(df[TEXT_COLUMN].duplicated().sum())

print("\nKategorien in category_parent:")
print(df["category_parent"].value_counts(dropna=False))

print("\nKategorien in category_child:")
print(df["category_child"].value_counts(dropna=False).head(25))

print("\nTextlängen in Wörtern:")
word_counts_raw = df[TEXT_COLUMN].dropna().astype(str).str.split().str.len()
print(word_counts_raw.describe())

print("\nBeispiele aus text_raw:")
for i, text in enumerate(df[TEXT_COLUMN].dropna().astype(str).head(5), start=1):
    print(f"\nBeispiel {i}:")
    print(text[:500])


In [ ]:
# Schritt 05 Arbeitsdatensatz erstellen

def prepare_records(
    source_df: pd.DataFrame,
    text_column: str,
    min_words: int,
    id_column: str,
):
    """Doppelte Datensätze entfernen und Meldungen mit unterschiedlichen IDs beibehalten."""
    if not source_df.columns.is_unique:
        raise ValueError("Die Spaltennamen müssen eindeutig sein.")
    missing_columns = [col for col in (text_column, id_column) if col not in source_df.columns]
    if missing_columns:
        raise ValueError(f"Erforderliche Spalten fehlen: {missing_columns}")
    if text_column == id_column:
        raise ValueError("Textspalte und ID-Spalte müssen unterschiedlich sein.")
    if not isinstance(min_words, int) or isinstance(min_words, bool) or min_words < 1:
        raise ValueError("min_words muss eine ganze Zahl größer oder gleich 1 sein.")

    counts = {"input": len(source_df)}

    # Fehlende IDs nicht als gemeinsame Meldung behandeln.
    id_keys = source_df[id_column].astype("string").str.strip()
    if id_keys.isna().any() or id_keys.eq("").any():
        raise ValueError("Meldungs-IDs fehlen. Die betroffenen Datensätze müssen geprüft werden.")

    # Nur über alle Spalten identische Datensätze entfernen, einschließlich der ID.
    work = source_df.drop_duplicates(keep="first").copy()
    counts["duplicates"] = len(source_df) - len(work)

    # Bei unterschiedlichen Inhalten zur selben ID keinen Datensatz willkürlich auswählen.
    if work[id_column].astype("string").str.strip().duplicated(keep=False).any():
        raise ValueError(
            "Zu einer Meldungs-ID liegen unterschiedliche Datensätze vor. "
            "Die Unterschiede müssen vor der Analyse geprüft werden."
        )

    before = len(work)
    work = work.dropna(subset=[text_column]).copy()
    counts["missing"] = before - len(work)
    work[text_column] = work[text_column].astype(str).str.strip()

    before = len(work)
    work = work.loc[work[text_column].str.len() > 0].copy()
    counts["empty"] = before - len(work)

    # Gleiche Texte mit unterschiedlichen IDs zählen weiterhin als einzelne Meldungen.
    work["word_count_raw"] = work[text_column].str.split().str.len()
    before = len(work)
    work = work.loc[work["word_count_raw"] >= min_words].copy()
    counts["raw_short"] = before - len(work)
    counts["remaining"] = len(work)
    counts["text_repetitions"] = int(work[text_column].duplicated().sum())

    if counts["input"] != sum(counts[key] for key in (
        "duplicates", "missing", "empty", "raw_short", "remaining"
    )):
        raise RuntimeError("Die dokumentierten Zeilenzahlen sind nicht konsistent.")

    return work.reset_index(drop=True), counts


df_work, record_counts = prepare_records(df, TEXT_COLUMN, MIN_WORDS_RAW, ID_COLUMN)

print("Ausgangszeilen:", record_counts["input"])
print("Entfernte vollständig identische Datensätze:", record_counts["duplicates"])
print("Entfernte fehlende Texte:", record_counts["missing"])
print("Entfernte leere Texte:", record_counts["empty"])
print("Entfernte Texte unter der Mindestlänge:", record_counts["raw_short"])
print("Meldungen im Arbeitsdatensatz:", record_counts["remaining"])
print("Weitere Meldungen mit gleichem Text, beibehalten:", record_counts["text_repetitions"])

# Unterschiedliche IDs mit gleichem Text bleiben erhalten, eine identische Zeile entfällt.
_test_data = pd.DataFrame({
    ID_COLUMN: [1, 2, 3, 4, 5, 6, 7, 8, 8],
    TEXT_COLUMN: [None, np.nan, pd.NA, "", "  ", " Müll ", "Müll", "Lampe defekt", "Lampe defekt"],
})
_test_work, _test_counts = prepare_records(_test_data, TEXT_COLUMN, 1, ID_COLUMN)
assert _test_work[ID_COLUMN].tolist() == [6, 7, 8]
assert _test_work[TEXT_COLUMN].tolist() == ["Müll", "Müll", "Lampe defekt"]
assert _test_counts == {
    "input": 9, "duplicates": 1, "missing": 3, "empty": 2,
    "raw_short": 0, "remaining": 3, "text_repetitions": 1,
}
print("Funktionstest für Fehlwerte und Dubletten: bestanden")

df_work[[ID_COLUMN, TEXT_COLUMN, "category_parent", "category_child", "word_count_raw"]].head(10)

In [ ]:
# Schritt 06 Ausgabeordner festlegen

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output-Ordner:")
print(OUTPUT_DIR)


In [ ]:
# Schritt 07 Text Preprocessing definieren

# Die Endungsregeln erkennen mögliche Ortsangaben, aber nicht alle Ortsnamen.
def is_location_token(token: str) -> bool:
    # Mögliche Straßennamen anhand der Endung erkennen und "straße" ausnehmen.
    if token == "straße":
        return False

    if token.endswith(("strasse", "straße", "str")) and len(token) > 5:
        return True

    # Mögliche Wegnamen erkennen und die folgenden allgemeinen Wegbegriffe ausnehmen.
    if token in {"gehweg", "radweg", "fahrradweg"}:
        return False

    if token.endswith("weg") and len(token) > 5:
        return True

    return False

def clean_text(text: str) -> str:
    text = str(text).lower()

    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []

    for token in text.split():
        token = CANONICAL_TERMS_FINAL.get(token, token)

        if is_location_token(token):
            continue

        if token not in STOPWORDS_FINAL and len(token) >= MIN_TOKEN_LENGTH:
            tokens.append(token)

    return " ".join(tokens)


In [ ]:
# Schritt 08 Text Preprocessing anwenden

# Für jeden Lauf eine neue Kopie des Arbeitsdatensatzes erzeugen.
df_model = df_work.copy()
df_model["text_clean"] = df_model[TEXT_COLUMN].apply(clean_text)
df_model["word_count_clean"] = df_model["text_clean"].str.split().str.len()

before_clean_filter = len(df_model)
df_model = df_model[df_model["word_count_clean"] >= MIN_TOKENS_CLEAN].copy()
after_clean_filter = len(df_model)
df_model = df_model.reset_index(drop=True)

print("Zeilen vor Textfilter:", before_clean_filter)
print("Zeilen nach Textfilter:", after_clean_filter)
print("Entfernte Meldungen:", before_clean_filter - after_clean_filter)
print("\nVerteilung der Wortanzahl nach Text Preprocessing:")
print(df_model["word_count_clean"].describe())


In [ ]:
# Schritt 09 Bereinigte Texte anhand von Beispielen ansehen

print("Beispiele vorher/nachher:")

for _, row in df_model[[TEXT_COLUMN, "text_clean"]].head(10).iterrows():
    print("\nRAW:")
    print(row[TEXT_COLUMN][:300])
    print("CLEAN:")
    print(row["text_clean"])


In [ ]:
# Schritt 10 Texte mit CountVectorizer und TfidfVectorizer vektorisieren

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts_clean = df_model["text_clean"].tolist()

count_vectorizer = CountVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    token_pattern=TOKEN_PATTERN,
    ngram_range=NGRAM_RANGE
)

tfidf_vectorizer = TfidfVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    token_pattern=TOKEN_PATTERN,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=SUBLINEAR_TF
)

X_count = count_vectorizer.fit_transform(texts_clean)
X_tfidf = tfidf_vectorizer.fit_transform(texts_clean)

print("Anzahl Dokumente:", len(texts_clean))
print("Count-Matrix:", X_count.shape)
print("TF-IDF-Matrix:", X_tfidf.shape)

print("\nBeispiele aus Count-Vokabular:")
print(count_vectorizer.get_feature_names_out()[:30])

print("\nBeispiele aus TF-IDF-Vokabular:")
print(tfidf_vectorizer.get_feature_names_out()[:30])


In [ ]:
# Kontrolle vor der Modellierung

print("Eingelesene Meldungen:", len(df))
print("Meldungen nach Text Preprocessing:", len(df_model))
print("Count-Matrix:", X_count.shape)
print("TF-IDF-Matrix:", X_tfidf.shape)
print(
    "Meldungen ohne Merkmale in der Count-Matrix:",
    int((X_count.getnnz(axis=1) == 0).sum())
)
print(
    "Meldungen ohne Merkmale in der TF-IDF-Matrix:",
    int((X_tfidf.getnnz(axis=1) == 0).sum())
)

# Kontrollieren, dass nach dem Preprocessing keine leeren Texte verbleiben.
assert df_model[TEXT_COLUMN].notna().all()
assert (df_model["word_count_clean"] >= MIN_TOKENS_CLEAN).all()
assert len(df_model) == X_count.shape[0] == X_tfidf.shape[0]

# Kurze, inhaltliche Testmeldungen bleiben mit der neuen Mindestlänge erhalten.
assert len(clean_text("Lampe defekt").split()) >= MIN_TOKENS_CLEAN
assert len(clean_text("Müll").split()) >= MIN_TOKENS_CLEAN
assert clean_text("Bitte danke") == ""
assert clean_text("Leuchte defekt") == "beleuchtung"
assert clean_text("Die Leuchten leuchten nicht") == ""  # Die Form "leuchten" bleibt unverändert.
print("Kontrolle der Textfilter: bestanden")



In [ ]:
# Schritt 10b Meldungen ohne Merkmale erfassen

def select_model_documents(documents, count_matrix, tfidf_matrix):
    """Die gleiche Auswahl von Meldungen auf beide Merkmalsmatrizen anwenden."""
    if len(documents) != count_matrix.shape[0] or count_matrix.shape != tfidf_matrix.shape:
        raise ValueError("Meldungen und Merkmalsmatrizen passen nicht zusammen.")
    if not documents.index.is_unique:
        raise ValueError("Der Index der Meldungen muss eindeutig sein.")

    count_matrix = count_matrix.tocsr(copy=True)
    tfidf_matrix = tfidf_matrix.tocsr(copy=True)
    for matrix in (count_matrix, tfidf_matrix):
        if not np.isfinite(matrix.data).all() or (matrix.data < 0).any():
            raise ValueError("Merkmalsmatrizen müssen endliche, nichtnegative Werte enthalten.")
        matrix.eliminate_zeros()

    count_mask = np.asarray(count_matrix.getnnz(axis=1)).ravel() > 0
    tfidf_mask = np.asarray(tfidf_matrix.getnnz(axis=1)).ravel() > 0
    if not np.array_equal(count_mask, tfidf_mask):
        raise ValueError("Count und TF-IDF ergeben unterschiedliche Meldungen ohne Merkmale.")

    # Die ursprünglichen Zeilenpositionen bleiben über den Index nachvollziehbar.
    fit_documents = documents.iloc[np.flatnonzero(count_mask)].copy()
    without_features = documents.iloc[np.flatnonzero(~count_mask)].copy()
    return (
        count_mask,
        fit_documents,
        count_matrix[count_mask],
        tfidf_matrix[count_mask],
        without_features,
    )


if not np.array_equal(
    count_vectorizer.get_feature_names_out(), tfidf_vectorizer.get_feature_names_out()
):
    raise ValueError("Count und TF-IDF verwenden unterschiedliche Vokabulare.")

model_mask, df_fit, X_count_fit, X_tfidf_fit, df_without_features = select_model_documents(
    df_model, X_count, X_tfidf
)
N_ANALYSIS_DOCS = len(df_model)
if N_ANALYSIS_DOCS == 0:
    raise ValueError("Nach dem Text Preprocessing sind keine Meldungen vorhanden.")

# Vokabular und IDF bleiben aus dem gesamten Korpus nach Text Preprocessing erhalten.
# Nur für das Training werden die Zeilen ohne Merkmale ausgeschlossen.
assert len(df_fit) + len(df_without_features) == N_ANALYSIS_DOCS
assert len(df_fit) == X_count_fit.shape[0] == X_tfidf_fit.shape[0]
assert (X_count_fit.getnnz(axis=1) > 0).all()
assert (X_tfidf_fit.getnnz(axis=1) > 0).all()

print("Meldungen nach Text Preprocessing:", N_ANALYSIS_DOCS)
print("Meldungen mit Merkmalen für das Modelltraining:", len(df_fit))
print("Nicht zugeordnet, da keine Merkmale im Vokabular:", len(df_without_features))
print("Count-Matrix für das Modelltraining:", X_count_fit.shape)
print("TF-IDF-Matrix für das Modelltraining:", X_tfidf_fit.shape)
print("Meldungen ohne Merkmale in den Trainingsmatrizen:", 0)
print("Bezugsgröße der Prozentanteile:", N_ANALYSIS_DOCS)
print("Kontrolle der Auswahl: bestanden")


In [ ]:
# Schritt 11 LDA und NMF trainieren

from sklearn.decomposition import LatentDirichletAllocation, NMF

if len(df_fit) == 0:
    raise ValueError("Keine Meldungen mit Merkmalen vorhanden. Daten und Vokabularfilter prüfen.")
if not isinstance(N_TOPICS, int) or isinstance(N_TOPICS, bool) or N_TOPICS < 1:
    raise ValueError("N_TOPICS muss eine positive ganze Zahl sein.")
if N_TOPICS > min(X_tfidf_fit.shape):
    raise ValueError("Für diese NMF-Initialisierung gibt es zu wenige Meldungen oder Merkmale.")

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    learning_method="batch",
    doc_topic_prior=LDA_DOC_TOPIC_PRIOR,
    topic_word_prior=LDA_TOPIC_WORD_PRIOR,
    max_iter=LDA_MAX_ITER
)

nmf_model = NMF(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    init="nndsvda",
    max_iter=NMF_MAX_ITER
)

lda_doc_topic = lda_model.fit_transform(X_count_fit)
nmf_doc_topic_raw = nmf_model.fit_transform(X_tfidf_fit)

print("LDA Dokument-Topic-Matrix:", lda_doc_topic.shape)
print("NMF Dokument-Topic-Matrix:", nmf_doc_topic_raw.shape)


In [ ]:
# Schritt 11a NMF-Koeffizienten für einheitliche Topic-Vektoren berechnen

def normalize_nmf_for_assignment(coefficients, components):
    """Topic-Vektoren auf L2-Norm 1 bringen und die Koeffizienten entsprechend anpassen."""
    w = np.asarray(coefficients, dtype=float)
    h = np.asarray(components, dtype=float)
    if w.ndim != 2 or h.ndim != 2 or w.shape[1] != h.shape[0] or h.shape[0] == 0:
        raise ValueError("Die beiden NMF-Matrizen passen nicht zusammen.")
    if h.shape[1] == 0:
        raise ValueError("Die Topic-Vektoren enthalten keine Merkmale.")
    if any(not np.isfinite(a).all() or (a < 0).any() for a in (w, h)):
        raise ValueError("NMF-Koeffizienten müssen endlich und nicht negativ sein.")

    lengths = np.linalg.norm(h, axis=1)
    if not np.isfinite(lengths).all() or (lengths <= 0).any():
        raise ValueError("Mindestens ein Topic-Vektor hat keine verwendbare L2-Norm.")
    h_unit = h / lengths[:, None]
    w_scaled = w * lengths[None, :]
    if not np.isfinite(w_scaled).all():
        raise ValueError("Die angepassten NMF-Koeffizienten enthalten ungültige Werte.")
    if not np.allclose(np.linalg.norm(h_unit, axis=1), 1.0):
        raise RuntimeError("Die Topic-Vektoren haben nicht die erwartete L2-Norm.")

    # Das Produkt beider Matrizen muss erhalten bleiben, kleine Abweichungen in der Rundung sind zulässig.
    for start in range(0, len(w), 256):
        before = w[start:start + 256] @ h
        after = w_scaled[start:start + 256] @ h_unit
        if not np.allclose(before, after, rtol=1e-10, atol=1e-12):
            raise RuntimeError("Die Rekonstruktion wurde durch die Skalierung verändert.")
    return w_scaled, h_unit


# Gleiche Rekonstruktion bei unterschiedlicher Skalierung muss gleiche angepasste Werte ergeben.
_scale_test_w = np.array([[1.0, 2.0], [0.0, 0.0]])
_scale_test_h = np.array([[10.0, 0.0], [0.0, 1.0]])
_scale_test_a, _ = normalize_nmf_for_assignment(_scale_test_w, _scale_test_h)
_scale_test_factors = np.array([0.01, 100.0])
_scale_test_b, _ = normalize_nmf_for_assignment(
    _scale_test_w / _scale_test_factors,
    _scale_test_h * _scale_test_factors[:, None],
)
if not np.allclose(_scale_test_a, _scale_test_b) or np.any(_scale_test_a[1] != 0):
    raise RuntimeError("Der Funktionstest der NMF-Skalierung ist fehlgeschlagen.")

# Die ursprünglichen Komponenten und Koeffizienten bleiben unverändert erhalten.
nmf_doc_topic, nmf_components_unit = normalize_nmf_for_assignment(
    nmf_doc_topic_raw, nmf_model.components_
)
print("NMF-Zuordnung verwendet Koeffizienten für L2-normierte Topic-Vektoren.")
print("Technische Kontrolle der NMF-Skalierung: bestanden")


In [ ]:
# Schritt 11b Topic-Zuordnung und nicht zugeordnete Meldungen erfassen

def assign_topics(doc_topic_matrix, model_mask, source_index):
    """Modellgewichte den ursprünglichen Zeilen zuordnen und fehlende Merkmale kennzeichnen."""
    weights = np.asarray(doc_topic_matrix, dtype=float)
    mask = np.asarray(model_mask)
    if mask.ndim != 1 or mask.dtype != bool or len(mask) != len(source_index):
        raise ValueError("Die Zeilenauswahl passt nicht zum Index der Meldungen.")
    if not source_index.is_unique:
        raise ValueError("Der Index der Meldungen muss eindeutig sein.")
    if weights.ndim != 2 or weights.shape[0] != int(mask.sum()) or weights.shape[1] == 0:
        raise ValueError("Die Dokument-Topic-Matrix passt nicht zur Zeilenauswahl.")
    if not np.isfinite(weights).all() or (weights < 0).any():
        raise ValueError("Topic-Gewichte müssen endlich und nichtnegativ sein.")

    result = pd.DataFrame(index=source_index.copy())
    result["Topic"] = pd.Series(pd.NA, index=source_index, dtype="Int64")
    result["Status"] = pd.Series(
        "Keine Merkmale im Vokabular", index=source_index, dtype="string"
    )
    result["Gleicher Höchstwert"] = False
    positions = np.flatnonzero(mask)
    if len(positions) == 0:
        return result

    max_weights = weights.max(axis=1)
    positive = max_weights > 0
    valid_index = source_index.take(positions[positive])
    zero_index = source_index.take(positions[~positive])

    # Nur positive Gewichte für eine Zuordnung verwenden.
    # Bei exakt gleichen Höchstwerten wählt argmax die erste Topic-Nummer.
    labels = weights[positive].argmax(axis=1) + 1
    result.loc[valid_index, "Topic"] = labels
    result.loc[valid_index, "Status"] = "Topic zugeordnet"
    result.loc[zero_index, "Status"] = "Keine positiven Topic-Gewichte"
    tied = (weights[positive] == max_weights[positive, None]).sum(axis=1) > 1
    result.loc[valid_index, "Gleicher Höchstwert"] = tied
    return result


lda_assignments = assign_topics(lda_doc_topic, model_mask, df_model.index)
nmf_assignments = assign_topics(nmf_doc_topic, model_mask, df_model.index)

# Künstlicher Test: eine zugeordnete Meldung, eine ohne Merkmale und eine ohne positive Gewichte.
_test_assignments = assign_topics(
    np.array([[0.2, 0.8], [0.0, 0.0]]),
    np.array([True, False, True]),
    pd.Index([10, 20, 30]),
)
assert _test_assignments.loc[10, "Topic"] == 2
assert _test_assignments.loc[[20, 30], "Topic"].isna().all()
assert _test_assignments.loc[20, "Status"] == "Keine Merkmale im Vokabular"
assert _test_assignments.loc[30, "Status"] == "Keine positiven Topic-Gewichte"

status_rows = []
for method, assignments in (("LDA", lda_assignments), ("NMF", nmf_assignments)):
    if not assignments.index.equals(df_model.index):
        raise RuntimeError("Die Reihenfolge der Zuordnungen passt nicht zu den Meldungen.")
    if assignments.loc[~model_mask, "Topic"].notna().any():
        raise RuntimeError("Eine Meldung ohne Merkmale hat unzulässigerweise ein Topic erhalten.")
    assigned_count = int(assignments["Topic"].notna().sum())
    unassigned_count = int(assignments["Topic"].isna().sum())
    assert assigned_count + unassigned_count == N_ANALYSIS_DOCS
    status_rows.append({
        "Verfahren": method,
        "Meldungen nach Text Preprocessing": N_ANALYSIS_DOCS,
        "Topic zugeordnet": assigned_count,
        "Nicht zugeordnet": unassigned_count,
        "Davon ohne Merkmale": int((assignments["Status"] == "Keine Merkmale im Vokabular").sum()),
        "Davon ohne positive Topic-Gewichte": int((assignments["Status"] == "Keine positiven Topic-Gewichte").sum()),
        "Anteil nicht zugeordnet in Prozent": round(100 * unassigned_count / N_ANALYSIS_DOCS, 1),
        "Mehrere Topics mit gleichem Höchstwert": int(assignments["Gleicher Höchstwert"].sum()),
    })
    print(f"{method}: {assigned_count} zugeordnet, {unassigned_count} nicht zugeordnet")

assignment_summary = pd.DataFrame(status_rows)
display(assignment_summary)
print("Kontrolle der Zuordnung: bestanden")


## Aktueller Modellvergleich

Der Vergleich verwendet die ergänzte Zuordnung `leuchte → beleuchtung` und bei allen NMF-Modellen dieselbe L2-Skalierung für die Zuordnung.
Kohärenzwerte und Meldungshäufigkeiten werden neu berechnet. Die endgültige Modellwahl erfolgt nach der inhaltlichen Prüfung der zugeordneten Originalmeldungen.


In [ ]:
# Schritt 11d verschiedene Anzahl von Topics vergleichen

import warnings
from pathlib import Path

import gensim
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning

# Diese Einstellungen gelten nur für den Vergleich
TOPIC_COUNTS_TO_COMPARE = [6, 8, 10, 12, 15]
COHERENCE_TOP_N = 10
COHERENCE_WINDOW_SIZE = 110


def compare_topic_counts(
    documents, fit_documents, count_matrix, tfidf_matrix, selected_rows,
    count_features, tfidf_features, lda_template, nmf_template,
    topic_counts, top_n, window_size,
):
    """Topic-Modelle auf derselben Datengrundlage trainieren und mit c_v vergleichen."""
    if count_matrix.shape != tfidf_matrix.shape:
        raise ValueError("Count und TF-IDF müssen dieselbe Form haben.")
    if len(fit_documents) != count_matrix.shape[0] or len(fit_documents) == 0:
        raise ValueError("Die Merkmalsmatrizen passen nicht zu den Meldungen für das Training.")
    if not np.array_equal(count_features, tfidf_features):
        raise ValueError("Count und TF-IDF müssen dasselbe Vokabular verwenden.")
    if len(count_features) != count_matrix.shape[1]:
        raise ValueError("Das Vokabular passt nicht zur Zahl der Merkmale.")
    mask = np.asarray(selected_rows)
    if mask.ndim != 1 or mask.dtype != bool or len(mask) != len(documents):
        raise ValueError("Die Auswahl der Meldungen ist ungültig.")
    if not documents.index.is_unique or not documents.index[mask].equals(fit_documents.index):
        raise ValueError("Die Reihenfolge der Meldungen stimmt nicht mit der Auswahl überein.")
    if not documents.loc[mask, "text_clean"].equals(fit_documents["text_clean"]):
        raise ValueError("Die Texte für das Training stimmen nicht mit der Auswahl überein.")
    for matrix in (count_matrix, tfidf_matrix):
        if not np.isfinite(matrix.data).all() or (matrix.data < 0).any():
            raise ValueError("Die Matrizen müssen endliche, nicht negative Werte enthalten.")
        if (np.asarray((matrix != 0).sum(axis=1)).ravel() == 0).any():
            raise ValueError("Die Trainingsmatrizen enthalten Meldungen ohne Merkmale.")
    if not topic_counts or len(set(topic_counts)) != len(topic_counts):
        raise ValueError("Die Anzahl der Topics muss eindeutig vorgegeben sein.")
    if any(isinstance(k, bool) or not isinstance(k, int) or k < 2 for k in topic_counts):
        raise ValueError("Für jeden Versuch sind mindestens zwei Topics erforderlich.")
    if max(topic_counts) > min(tfidf_matrix.shape):
        raise ValueError("Für die NMF-Initialisierung gibt es zu wenige Meldungen oder Merkmale.")
    if isinstance(top_n, bool) or not isinstance(top_n, int) or not 2 <= top_n <= len(count_features):
        raise ValueError("Die Anzahl der Top-Begriffe passt nicht zum Vokabular.")
    if isinstance(window_size, bool) or not isinstance(window_size, int) or window_size < 2:
        raise ValueError("Die Fenstergröße für c_v muss mindestens zwei betragen.")
    if tuple(count_vectorizer.ngram_range) != (1, 1):
        raise ValueError("Dieser Kohärenzvergleich ist für die aktuelle Unigramm-Darstellung ausgelegt.")

    # Alle Varianten verwenden dieselben bereinigten Texte für die Kohärenzberechnung.
    reference_texts = documents["text_clean"].str.split().tolist()
    if any(not isinstance(text, list) or not text for text in reference_texts):
        raise ValueError("Die Referenztexte für c_v dürfen nicht fehlen oder leer sein.")
    dictionary = Dictionary(reference_texts)
    missing_terms = set(count_features) - set(dictionary.token2id)
    if missing_terms:
        raise ValueError("Begriffe aus dem Vokabular fehlen in den Referenztexten für c_v.")

    # Die effektiven LDA-Priors bleiben wie im bisherigen Modell, statt sich mit 1/k zu ändern.
    lda_priors = {
        "doc_topic_prior": float(lda_template.doc_topic_prior_),
        "topic_word_prior": float(lda_template.topic_word_prior_),
    }
    if any(not np.isfinite(value) or value <= 0 for value in lda_priors.values()):
        raise ValueError("Die LDA-Priors müssen positiv und endlich sein.")

    models, weights_by_model, assignments_by_model = {}, {}, {}
    summary_rows, topic_rows = [], []
    print("Meldungen für c_v:", len(documents))
    print("Meldungen je Modelltraining:", len(fit_documents))
    print("Merkmale:", count_matrix.shape[1])
    print("Anzahl der Topics:", topic_counts)
    print("Top-Begriffe je Topic für c_v:", top_n)
    print("Feste LDA-Priors:", lda_priors)

    total_runs = 2 * len(topic_counts)
    completed = 0
    for method, template, matrix in (
        ("LDA", lda_template, count_matrix),
        ("NMF", nmf_template, tfidf_matrix),
    ):
        for k in topic_counts:
            print(f"[{completed + 1}/{total_runs}] {method} mit {k} Topics", flush=True)
            candidate = clone(template).set_params(n_components=k)
            if method == "LDA":
                candidate.set_params(**lda_priors)
            with warnings.catch_warnings(record=True) as recorded_warnings:
                warnings.simplefilter("always", ConvergenceWarning)
                weights = candidate.fit_transform(matrix)
            warning_messages = [str(warning.message) for warning in recorded_warnings]
            for message in warning_messages:
                print(f"  Warnung: {message}")

            components = candidate.components_
            if components.shape != (k, len(count_features)):
                raise RuntimeError("Die Modell-Komponenten passen nicht zum Vergleich.")
            if not np.isfinite(components).all() or (components < 0).any():
                raise RuntimeError("Das Modell enthält ungültige Gewichte.")
            if ((components > 0).sum(axis=1) < top_n).any():
                raise RuntimeError("Ein Topic enthält zu wenige positive Gewichte für die Top-Begriffe.")
            top_indices = np.argsort(-components, axis=1, kind="stable")[:, :top_n]
            topics = [count_features[indices].tolist() for indices in top_indices]

            # Ein Prozess vermeidet zusätzliche Worker bei der Ausführung unter Windows.
            coherence = CoherenceModel(
                topics=topics, texts=reference_texts, dictionary=dictionary,
                coherence="c_v", topn=top_n, window_size=window_size, processes=1,
            )
            per_topic = np.asarray(coherence.get_coherence_per_topic(), dtype=float)
            if per_topic.shape != (k,) or not np.isfinite(per_topic).all():
                raise RuntimeError(f"Für {method} mit {k} Topics ist c_v nicht vollständig berechenbar.")

            # Alle NMF-Varianten verwenden dieselbe Skalierung für die Zuordnung.
            # Das Modell und die für c_v verwendeten Top-Begriffe bleiben dabei unverändert.
            if method == "NMF":
                assignment_weights, _ = normalize_nmf_for_assignment(weights, components)
                assignment_rule = "NMF mit L2-normierten Topic-Vektoren"
            else:
                assignment_weights = weights
                assignment_rule = "LDA-Themenanteile"
            assignments = assign_topics(assignment_weights, mask, documents.index)
            if assignments.loc[~mask, "Topic"].notna().any():
                raise RuntimeError("Eine Meldung ohne Merkmale wurde einem Topic zugeordnet.")
            counts = assignments["Topic"].value_counts().reindex(range(1, k + 1), fill_value=0)
            assigned = int(assignments["Topic"].notna().sum())
            unassigned = int(assignments["Topic"].isna().sum())
            if int(counts.sum()) != assigned or assigned + unassigned != len(documents):
                raise RuntimeError("Die Häufigkeiten des Vergleichs sind nicht konsistent.")

            key = (method, k)
            models[key] = candidate
            # Diese Werte müssen zu den gespeicherten Topic-Zuordnungen passen.
            weights_by_model[key] = assignment_weights
            assignments_by_model[key] = assignments
            summary_rows.append({
                "Verfahren": method,
                "Anzahl Topics": k,
                "c_v Mittelwert": float(per_topic.mean()),
                "c_v Minimum": float(per_topic.min()),
                "Zugeordnet": assigned,
                "Nicht zugeordnet": unassigned,
                "Topics ohne Zuordnung": int(counts.eq(0).sum()),
                "Iterationen": int(candidate.n_iter_),
                "Warnungen": len(warning_messages),
                "Hinweise": " | ".join(warning_messages),
                "Regel der Zuordnung": assignment_rule,
            })
            for position, (terms, value) in enumerate(zip(topics, per_topic), start=1):
                topic_rows.append({
                    "Verfahren": method,
                    "Anzahl Topics": k,
                    "Topic": position,
                    "Anzahl Meldungen": int(counts.loc[position]),
                    "Anteil Prozent": 100 * int(counts.loc[position]) / len(documents),
                    "Bezugsmenge Meldungen": len(documents),
                    "c_v": float(value),
                    "Top-Begriffe": ", ".join(terms),
                    "Regel der Zuordnung": assignment_rule,
                })
            completed += 1
            print(f"  c_v Mittelwert: {per_topic.mean():.4f}, nicht zugeordnet: {unassigned}", flush=True)

    return (
        pd.DataFrame(summary_rows), pd.DataFrame(topic_rows),
        models, weights_by_model, assignments_by_model,
    )


required_for_topic_comparison = (
    "df_model", "df_fit", "model_mask", "X_count_fit", "X_tfidf_fit",
    "count_vectorizer", "tfidf_vectorizer", "lda_model", "nmf_model",
    "assign_topics", "normalize_nmf_for_assignment", "OUTPUT_DIR",
)
missing_for_topic_comparison = [
    name for name in required_for_topic_comparison if name not in globals()
]
if missing_for_topic_comparison:
    raise RuntimeError(
        "Zuerst alle Zellen bis einschließlich Schritt 11b ausführen. "
        f"Es fehlen: {', '.join(missing_for_topic_comparison)}"
    )

(
    topic_count_review, topic_count_details,
    topic_count_models, topic_count_weights, topic_count_assignments,
) = compare_topic_counts(
    df_model, df_fit, X_count_fit, X_tfidf_fit, model_mask,
    count_vectorizer.get_feature_names_out(), tfidf_vectorizer.get_feature_names_out(),
    lda_model, nmf_model, TOPIC_COUNTS_TO_COMPARE, COHERENCE_TOP_N, COHERENCE_WINDOW_SIZE,
)

print("\nVergleich der Anzahl von Topics")
display(topic_count_review.drop(columns=["Hinweise"]).round(4))

# Nur Vergleichswerte und Top-Begriffe exportieren, keine originalen Meldungen.
comparison_dir = Path(OUTPUT_DIR)
comparison_dir.mkdir(parents=True, exist_ok=True)
comparison_path = comparison_dir / "vergleich_topic_anzahl.csv"
comparison_topics_path = comparison_dir / "vergleich_topic_anzahl_details.csv"
topic_count_review.to_csv(comparison_path, index=False, sep=";", encoding="utf-8-sig")
topic_count_details.to_csv(comparison_topics_path, index=False, sep=";", encoding="utf-8-sig")

print("Gespeichert:")
print(comparison_path.resolve())
print(comparison_topics_path.resolve())
print("Technische Kontrolle des Vergleichs: bestanden")
print("Die Modelle und Zuordnungen der Hauptanalyse sind unverändert.")

# Einstellungen mit den Ergebnissen speichern, damit unterschiedliche Läufe erkennbar bleiben.
import hashlib
import json
import sklearn

comparison_settings = {
    "csv_sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(),
    "leuchte_zuordnung": CANONICAL_TERMS_FINAL.get("leuchte"),
    "nmf_zuordnung": "L2-normierte Topic-Vektoren",
    "anzahl_topics": list(TOPIC_COUNTS_TO_COMPARE),
    "anzahl_meldungen_nach_preprocessing": len(df_model),
    "anzahl_meldungen_training": len(df_fit),
    "anzahl_merkmale": X_count_fit.shape[1],
    "min_df": MIN_DF,
    "max_df": MAX_DF,
    "ngram_range": list(NGRAM_RANGE),
    "token_pattern": TOKEN_PATTERN,
    "sublinear_tf": SUBLINEAR_TF,
    "min_words_raw": MIN_WORDS_RAW,
    "min_tokens_clean": MIN_TOKENS_CLEAN,
    "min_token_length": MIN_TOKEN_LENGTH,
    "stopwoerter": sorted(STOPWORDS_FINAL),
    "begriffszuordnungen": dict(CANONICAL_TERMS_FINAL),
    "random_state": RANDOM_STATE,
    "lda_doc_topic_prior": LDA_DOC_TOPIC_PRIOR,
    "lda_topic_word_prior": LDA_TOPIC_WORD_PRIOR,
    "lda_max_iter": LDA_MAX_ITER,
    "nmf_max_iter": NMF_MAX_ITER,
    "coherence": "c_v",
    "coherence_top_n": COHERENCE_TOP_N,
    "coherence_window_size": COHERENCE_WINDOW_SIZE,
    "gensim": gensim.__version__,
    "scikit_learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
settings_path = comparison_dir / "vergleich_topic_anzahl_einstellungen.json"
settings_path.write_text(json.dumps(comparison_settings, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


In [ ]:
# Schritt 11e Meldungen ausgewählter Modelle zur inhaltlichen Prüfung zusammenstellen

from pathlib import Path

import numpy as np
import pandas as pd

# Die Kandidaten werden nur geprüft, die Hauptanalyse bleibt unverändert.
MODELS_TO_REVIEW = [("NMF", 12), ("NMF", 15), ("LDA", 10), ("LDA", 15)]
EXAMPLES_BY_WEIGHT = 2
RANDOM_EXAMPLES = 3


def collect_topic_examples(
    documents, fit_documents, selected_rows, models, weights_by_model,
    assignments_by_model, details, candidates, text_column, id_column,
    n_by_weight=2, n_random=3, random_state=42,
):
    """Pro Topic die stark gewichteten Meldungen und eine getrennte Zufallsauswahl zusammenstellen."""
    required_columns = {text_column, id_column, "category_parent", "category_child"}
    if not required_columns.issubset(documents.columns):
        raise ValueError("Textspalte, ID oder Kategorien fehlen in den Meldungen.")
    if not documents.index.is_unique or not fit_documents.index.is_unique:
        raise ValueError("Die Indizes der Meldungen müssen eindeutig sein.")
    if documents[id_column].isna().any() or documents[id_column].duplicated().any():
        raise ValueError("Die Melde-IDs müssen vorhanden und eindeutig sein.")
    if any(isinstance(n, bool) or not isinstance(n, int) or n < 0 for n in (n_by_weight, n_random)):
        raise ValueError("Die Anzahl der Beispiele muss jeweils eine nicht negative ganze Zahl sein.")
    if n_by_weight + n_random == 0:
        raise ValueError("Mindestens eine Meldung pro Topic muss ausgewählt werden.")
    if not candidates or len(set(candidates)) != len(candidates):
        raise ValueError("Die Liste der zu prüfenden Modelle darf nicht leer sein oder Einträge wiederholen.")

    mask = np.asarray(selected_rows)
    if mask.ndim != 1 or mask.dtype != bool or len(mask) != len(documents):
        raise ValueError("Die Auswahl der Meldungen passt nicht zum Datensatz.")
    if not documents.index[mask].equals(fit_documents.index):
        raise ValueError("Die Reihenfolge der Meldungen stimmt nicht mit den Modellgewichten überein.")
    if not documents.loc[mask, "text_clean"].equals(fit_documents["text_clean"]):
        raise ValueError("Die Texte wurden seit dem Modellvergleich verändert.")

    rng = np.random.default_rng(random_state)
    rows = []
    overview = []
    fit_index = fit_documents.index

    for method, k in candidates:
        key = (method, k)
        if any(key not in collection for collection in (models, weights_by_model, assignments_by_model)):
            raise ValueError(f"Das Vergleichsmodell {method} mit {k} Topics fehlt. Zuerst Schritt 11d ausführen.")
        model = models[key]
        # Bei NMF sind dies die bereits angepassten Koeffizienten aus Schritt 11d.
        weights = np.asarray(weights_by_model[key], dtype=float)
        assignments = assignments_by_model[key]
        if model.n_components != k or weights.shape != (len(fit_documents), k):
            raise ValueError(f"Die Modellgewichte von {method} mit {k} Topics haben eine unerwartete Form.")
        if not np.isfinite(weights).all() or (weights < 0).any():
            raise ValueError("Die Modellgewichte müssen endlich und nicht negativ sein.")
        if not assignments.index.equals(documents.index):
            raise ValueError("Die Zuordnungen passen nicht zum Index der Meldungen.")
        if assignments.loc[~mask, "Topic"].notna().any():
            raise ValueError("Eine Meldung ohne Merkmale hat eine Topic-Zuordnung.")

        # Die gespeicherten Zuordnungen müssen zu den vorhandenen Gewichtungen passen.
        expected = pd.Series(pd.NA, index=fit_index, dtype="Int64")
        positive = weights.max(axis=1) > 0
        expected.loc[fit_index[positive]] = weights[positive].argmax(axis=1) + 1
        actual = assignments.loc[fit_index, "Topic"].astype("Int64")
        if not np.array_equal(expected.to_numpy(na_value=-1), actual.to_numpy(na_value=-1)):
            raise ValueError("Modellgewichte und Topic-Zuordnungen stimmen nicht überein.")

        model_details = details.loc[
            (details["Verfahren"] == method) & (details["Anzahl Topics"] == k)
        ].set_index("Topic")
        if not model_details.index.is_unique or set(model_details.index) != set(range(1, k + 1)):
            raise ValueError("Die Detailtabelle enthält nicht genau einen Eintrag pro Topic.")

        before_model = len(rows)
        for topic in range(1, k + 1):
            positions = np.flatnonzero(actual.eq(topic).fillna(False).to_numpy(dtype=bool))
            detail = model_details.loc[topic]
            if len(positions) != int(detail["Anzahl Meldungen"]):
                raise ValueError("Die Häufigkeiten der Detailtabelle passen nicht zu den Zuordnungen.")

            # Hohe Gewichtungen dienen nur zur Auswahl von Beispielen, nicht als Beweis einer korrekten Zuordnung.
            order = np.argsort(-weights[positions, topic - 1], kind="stable")
            strongest = positions[order[:min(n_by_weight, len(positions))]]
            remaining = positions[~np.isin(positions, strongest)]
            random_selection = rng.choice(remaining, size=min(n_random, len(remaining)), replace=False)
            selections = [(int(p), "Hohes Topic-Gewicht") for p in strongest]
            selections += [(int(p), "Zufallsauswahl") for p in random_selection]
            if len({p for p, _ in selections}) != len(selections):
                raise RuntimeError("Eine Meldung wurde innerhalb eines Topics mehrfach ausgewählt.")

            for position, selection in selections:
                record = documents.loc[fit_index[position]]
                rows.append({
                    "Verfahren": method,
                    "Anzahl Topics": k,
                    "Topic": topic,
                    "Top-Begriffe": detail["Top-Begriffe"],
                    "c_v": float(detail["c_v"]),
                    "Meldungen im Topic": len(positions),
                    "Auswahl": selection,
                    "ID": record[id_column],
                    "Meldetext": record[text_column],
                    "Hauptkategorie": record["category_parent"],
                    "Unterkategorie": record["category_child"],
                })
        overview.append({"Verfahren": method, "Anzahl Topics": k, "Ausgewählte Meldungen": len(rows) - before_model})

    columns = [
        "Verfahren", "Anzahl Topics", "Topic", "Top-Begriffe", "c_v",
        "Meldungen im Topic", "Auswahl", "ID", "Meldetext", "Hauptkategorie", "Unterkategorie",
    ]
    return pd.DataFrame(rows, columns=columns), pd.DataFrame(overview)


required_for_examples = (
    "df_model", "df_fit", "model_mask", "topic_count_models", "topic_count_weights",
    "topic_count_assignments", "topic_count_details", "TEXT_COLUMN", "ID_COLUMN", "OUTPUT_DIR",
)
missing_for_examples = [name for name in required_for_examples if name not in globals()]
if missing_for_examples:
    raise RuntimeError(
        "Zuerst die Hauptanalyse bis Schritt 11b und den Modellvergleich in Schritt 11d ausführen. "
        f"Es fehlen: {', '.join(missing_for_examples)}"
    )

review_topic_examples, review_example_counts = collect_topic_examples(
    df_model, df_fit, model_mask,
    topic_count_models, topic_count_weights, topic_count_assignments, topic_count_details,
    MODELS_TO_REVIEW, TEXT_COLUMN, ID_COLUMN,
    n_by_weight=EXAMPLES_BY_WEIGHT, n_random=RANDOM_EXAMPLES, random_state=RANDOM_STATE,
)

# Die Datei enthält die Originalmeldungen und bleibt im lokalen Ausgabeordner.
example_path = Path(OUTPUT_DIR) / "pruefung_topic_meldungen.csv"
example_path.parent.mkdir(parents=True, exist_ok=True)
review_topic_examples.to_csv(example_path, index=False, sep=";", encoding="utf-8-sig")
print(review_example_counts.to_string(index=False))
print("Insgesamt ausgewählte Meldungen:", len(review_topic_examples))
print("Technische Kontrolle der Auswahl: bestanden")
print("Gespeichert:", example_path.resolve())


## Bewertung des aktuellen Modellvergleichs

NMF mit zwölf Topics erzielt den höchsten mittleren Kohärenzwert der
untersuchten Varianten (`c_v = 0,4201`) und wird für die weitere
Ergebnisdarstellung bevorzugt. Innerhalb der LDA-Varianten liegt
das Modell mit zehn Topics vorn (`c_v = 0,3705`).

Die Prüfung ausgeworiginaler Meldungenldungen zeigt klar erkennbare
Schwerpunkte, aber auch gemischte Topics und einzelne unpassende
Zuordnungen.

In [ ]:
# Schritt 11f Ausgewählte Modelle für die Ergebnisübersichten übernehmen

required_for_selection = (
    "SELECTED_TOPIC_COUNTS", "topic_count_models", "topic_count_weights",
    "topic_count_assignments", "topic_count_review", "topic_count_details",
    "df_model", "df_fit", "model_mask", "assign_topics", "OUTPUT_DIR",
)
missing_for_selection = [name for name in required_for_selection if name not in globals()]
if missing_for_selection:
    raise RuntimeError(
        "Zuerst die Konfiguration und den Modellvergleich ausführen. "
        f"Es fehlen: {', '.join(missing_for_selection)}"
    )
if set(SELECTED_TOPIC_COUNTS) != {"LDA", "NMF"}:
    raise ValueError("Für LDA und NMF muss jeweils eine Anzahl von Topics gewählt sein.")

selected_models, selected_weights, selected_assignments = {}, {}, {}
selection_rows, selected_status_rows = [], []

for method in ("LDA", "NMF"):
    k = SELECTED_TOPIC_COUNTS[method]
    if isinstance(k, bool) or not isinstance(k, int) or k < 2:
        raise ValueError("Die gewählte Anzahl von Topics muss eine ganze Zahl ab zwei sein.")
    key = (method, k)
    if any(key not in collection for collection in (
        topic_count_models, topic_count_weights, topic_count_assignments
    )):
        raise ValueError(f"Das Vergleichsmodell {method} mit {k} Topics fehlt.")

    model = topic_count_models[key]
    coefficients = np.asarray(topic_count_weights[key], dtype=float).copy()
    assignments = topic_count_assignments[key].copy(deep=True)
    if model.n_components != k or coefficients.shape != (len(df_fit), k):
        raise ValueError("Das Modell und die Topic-Matrix passen nicht zusammen.")

    components = np.asarray(model.components_, dtype=float)
    if (not np.isfinite(components).all() or (components < 0).any()
            or (np.linalg.norm(components, axis=1) <= 0).any()):
        raise ValueError("Die ausgewählten Topic-Vektoren enthalten ungültige Werte.")

    # Die Auswahl übernimmt genau die bereits geprüften Zuordnungen.
    expected_assignments = assign_topics(coefficients, model_mask, df_model.index)
    if not assignments.equals(expected_assignments):
        raise ValueError("Die gespeicherten Zuordnungen passen nicht zu den Koeffizienten.")

    result = topic_count_review.loc[
        (topic_count_review["Verfahren"] == method)
        & (topic_count_review["Anzahl Topics"] == k)
    ]
    details = topic_count_details.loc[
        (topic_count_details["Verfahren"] == method)
        & (topic_count_details["Anzahl Topics"] == k)
    ].set_index("Topic")
    if len(result) != 1 or not details.index.is_unique or set(details.index) != set(range(1, k + 1)):
        raise ValueError("Der gewählte Modellstand ist in den Vergleichstabellen nicht eindeutig vorhanden.")
    counts = assignments["Topic"].value_counts().reindex(range(1, k + 1), fill_value=0)
    if not np.array_equal(counts.to_numpy(), details.loc[range(1, k + 1), "Anzahl Meldungen"].to_numpy()):
        raise ValueError("Die Häufigkeiten stimmen nicht mit dem Modellvergleich überein.")

    assigned = int(assignments["Topic"].notna().sum())
    unassigned = int(assignments["Topic"].isna().sum())
    if assigned + unassigned != len(df_model) or assigned != int(result.iloc[0]["Zugeordnet"]):
        raise ValueError("Die Summen der Zuordnungen stimmen nicht mit dem Modellvergleich überein.")

    selected_models[method] = model
    selected_weights[method] = coefficients
    selected_assignments[method] = assignments
    selection_rows.append({
        "Verfahren": method, "Anzahl Topics": k,
        "c_v Mittelwert": float(result.iloc[0]["c_v Mittelwert"]),
        "Zugeordnet": assigned, "Nicht zugeordnet": unassigned,
    })
    selected_status_rows.append({
        "Verfahren": method, "Anzahl Topics": k,
        "Meldungen nach Text Preprocessing": len(df_model),
        "Topic zugeordnet": assigned, "Nicht zugeordnet": unassigned,
        "Davon ohne Merkmale": int(assignments["Status"].eq("Keine Merkmale im Vokabular").sum()),
        "Davon ohne positive Topic-Gewichte": int(assignments["Status"].eq("Keine positiven Topic-Gewichte").sum()),
        "Anteil von nicht zugeordnet in Prozent": round(100 * unassigned / len(df_model), 1),
        "Mehrere Topics mit gleichem Höchstwert": int(assignments["Gleicher Höchstwert"].sum()),
    })

# Alle nachfolgenden Ausgaben verwenden diese Modelle und die dazugehörigen Zuordnungen.

lda_model, nmf_model = selected_models["LDA"], selected_models["NMF"]
lda_doc_topic, nmf_doc_topic = selected_weights["LDA"], selected_weights["NMF"]
lda_assignments, nmf_assignments = selected_assignments["LDA"], selected_assignments["NMF"]

# Die NMF-Koeffizienten aus dem Vergleich sind bereits auf L2-normierte Topics abgestimmt.
_nmf_lengths = np.linalg.norm(nmf_model.components_, axis=1)
nmf_components_unit = nmf_model.components_ / _nmf_lengths[:, None]
nmf_doc_topic_raw = nmf_doc_topic / _nmf_lengths[None, :]

N_ANALYSIS_DOCS = len(df_model)
assignment_summary = pd.DataFrame(selected_status_rows)
model_selection = pd.DataFrame(selection_rows)

print(model_selection.round({"c_v Mittelwert": 4}).to_string(index=False))
print("Bezugsgröße der Prozentanteile:", N_ANALYSIS_DOCS)
print("Technische Kontrolle der Modellauswahl: bestanden")

selection_path = OUTPUT_DIR / "modellauswahl.csv"
model_selection.to_csv(selection_path, index=False, sep=";", encoding="utf-8-sig")
print("Gespeichert:", selection_path.resolve())


In [ ]:
# Schritt 12 Wichtigste Begriffe je Topic anzeigen

def print_top_words(model, feature_names, n_top_words=12, title=""):
    print("\n" + title)
    print("=" * len(title))

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[::-1][:n_top_words]
        top_words = [feature_names[i] for i in top_indices]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

print_top_words(
    lda_model,
    count_vectorizer.get_feature_names_out(),
    n_top_words=12,
    title="LDA Topics auf CountVectorizer"
)

print_top_words(
    nmf_model,
    tfidf_vectorizer.get_feature_names_out(),
    n_top_words=12,
    title="NMF Topics auf TfidfVectorizer"
)


In [ ]:
# Schritt 13 Technische Themenübersichten erstellen

def build_topic_summary(model, feature_names, assignments, source_df, n_top_words=12, n_examples=2):
    if not assignments.index.equals(source_df.index):
        raise ValueError("Meldungen und Topic-Zuordnungen haben nicht denselben Index.")
    n_docs = len(source_df)
    if n_docs == 0:
        raise ValueError("Die Bezugsmenge für die Prozentanteile ist leer.")

    rows = []

    for topic_idx in range(model.n_components):
        topic_number = topic_idx + 1

        # Top Begriffe
        topic_weights = model.components_[topic_idx]
        top_indices = topic_weights.argsort()[::-1][:n_top_words]
        top_words = ", ".join(feature_names[top_indices])

        # Dokumente mit dominantem Topic
        doc_indices = np.flatnonzero(
            assignments["Topic"].eq(topic_number).fillna(False).to_numpy(dtype=bool)
        )
        doc_count = len(doc_indices)
        doc_share = round((doc_count / n_docs) * 100, 1)

        # Häufigste bestehende Kategorie zur Plausibilisierung
        if doc_count > 0:
            category_counts = (
                source_df.iloc[doc_indices]["category_parent"]
                .value_counts()
                .head(3)
            )
            top_categories = " | ".join(
                [f"{cat}: {count}" for cat, count in category_counts.items()]
            )

            # Erste zugeordnete Meldungen auswählen und kürzen.
            examples = " || ".join(
                source_df.iloc[doc_indices][TEXT_COLUMN]
                .astype(str)
                .head(n_examples)
                .str.slice(0, 180)
                .tolist()
            )
        else:
            top_categories = ""
            examples = ""

        rows.append({
            "Topic": topic_number,
            "Anzahl Meldungen": doc_count,
            "Anteil Prozent": doc_share,
            "Bezugsmenge Meldungen": n_docs,
            "Top Begriffe": top_words,
            "Häufigste Kategorie": top_categories,
            "Beispielmeldungen": examples
        })

    summary = pd.DataFrame(rows)
    summary = summary.sort_values("Anzahl Meldungen", ascending=False).reset_index(drop=True)
    summary.insert(0, "Rang", np.arange(1, len(summary) + 1))

    if int(summary["Anzahl Meldungen"].sum()) != int(assignments["Topic"].notna().sum()):
        raise RuntimeError("Die Themenhäufigkeiten stimmen nicht mit den Zuordnungen überein.")
    return summary

lda_summary = build_topic_summary(
    lda_model,
    count_vectorizer.get_feature_names_out(),
    lda_assignments,
    df_model
)

nmf_summary = build_topic_summary(
    nmf_model,
    tfidf_vectorizer.get_feature_names_out(),
    nmf_assignments,
    df_model
)

print("LDA Themenübersicht:")
display(lda_summary)

print("NMF Themenübersicht:")
display(nmf_summary)

print("Die Anteile beziehen sich auf alle", N_ANALYSIS_DOCS, "Meldungen nach Text Preprocessing.")
print("Nicht zugeordnete Meldungen sind in der separaten Zuordnungsübersicht enthalten.")
print("Gerundete Prozentanteile können sich geringfügig von 100 Prozent unterscheiden.")


In [ ]:
# Schritt 14 Technische Themenübersichten als CSV speichern



# Die Spalte mit Rohtextbeispielen nicht in diesen CSV-Export übernehmen.
# Beispielmeldungen können Adressen oder personenbezogene Hinweise enthalten.
lda_export = lda_summary.drop(columns=["Beispielmeldungen"], errors="ignore")
nmf_export = nmf_summary.drop(columns=["Beispielmeldungen"], errors="ignore")

lda_path = OUTPUT_DIR / "themenuebersicht_lda.csv"
nmf_path = OUTPUT_DIR / "themenuebersicht_nmf.csv"

lda_export.to_csv(lda_path, index=False, encoding="utf-8-sig")
nmf_export.to_csv(nmf_path, index=False, encoding="utf-8-sig")

print("Gespeichert:")
print(lda_path.resolve())
print(nmf_path.resolve())

assignment_path = OUTPUT_DIR / "zuordnung_lda_nmf.csv"
assignment_summary.to_csv(assignment_path, index=False, sep=";", encoding="utf-8-sig")
print(assignment_path.resolve())


In [ ]:
# Schritt 15 Die ausgewählten NMF-Topics beschriften

import numpy as np
import pandas as pd

# Die Angaben beziehen sich auf die geprüfte Stichprobe des Modells mit zwölf Topics.
# Die Beispiele sind sinngemäß zusammengefasst, keine wörtlichen Zitate.
# Gemischte Topics behalten ihre Nummer und erhalten keine zu enge Themenbezeichnung.
topic_descriptions = {
    1: {
        "Thema": "Beleuchtung",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Ausfälle und Schäden an der öffentlichen Beleuchtung.",
        "Beispielhafte Hinweise": "Laterne ausgefallen, Straßenlampe defekt",
        "Erwartete Begriffe": ("beleuchtung", "kurve", "licht"),
    },
    2: {
        "Thema": "Müll und Abfall",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Abgelagerter Müll und Abfälle an überfüllten Behältern.",
        "Beispielhafte Hinweise": "Müll abgestellt, Mülltonnen überfüllt",
        "Erwartete Begriffe": ("müll", "illegaler", "mülleimer"),
    },
    3: {
        "Thema": "Abgestellte Fahrräder",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Abgestellte, beschädigte oder längere Zeit ungenutzte Fahrräder.",
        "Beispielhafte Hinweise": "Defekte Fahrräder abgestellt, Fahrradrahmen gebrochen",
        "Erwartete Begriffe": ("fahrrad", "herrenloses", "sattel"),
    },
    4: {
        "Thema": "Gemischte Meldungen (Topic 4)",
        "Einordnung": "Gemischtes Topic",
        "Kurzbeschreibung": "Verschiedene Anliegen, darunter Müll, Beleuchtung und Straßenschäden.",
        "Beispielhafte Hinweise": "Möbel an der Straße, Laternen ausgefallen, Frostschäden",
        "Erwartete Begriffe": ("straße", "entlang", "abgemeldeter"),
    },
    5: {
        "Thema": "Gehwege und Einschränkungen",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Eingeschränkte Nutzung von Gehwegen durch Bewuchs, Gegenstände oder Sperrungen.",
        "Beispielhafte Hinweise": "Gehweg zugewachsen, Durchgang verengt, Baustelle auf dem Gehweg",
        "Erwartete Begriffe": ("gehweg", "wächst", "hecke"),
    },
    6: {
        "Thema": "Beschilderung",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Beschädigte, unleserliche oder umgestürzte Schilder.",
        "Beispielhafte Hinweise": "Schild umgeknickt, Straßenschild unkenntlich",
        "Erwartete Begriffe": ("schild", "umgefahren", "ausgeblichen"),
    },
    7: {
        "Thema": "Gemischte Meldungen (Topic 7)",
        "Einordnung": "Gemischtes Topic",
        "Kurzbeschreibung": "Scherben und weitere Anliegen zu Wegen, Pollern und Abdeckungen.",
        "Beispielhafte Hinweise": "Glasscherben auf dem Radweg, Poller herausgerissen, Gullydeckel lose",
        "Erwartete Begriffe": ("glasscherben", "radweg", "fahrradweg"),
    },
    8: {
        "Thema": "Ampeln",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Störungen von Ampeln und ihrer Bedienelemente.",
        "Beispielhafte Hinweise": "Ampeltaste defekt, Grünsignal bleibt aus",
        "Erwartete Begriffe": ("ampel", "fußgänger", "grün"),
    },
    9: {
        "Thema": "Gemischte Meldungen (Topic 9)",
        "Einordnung": "Gemischtes Topic",
        "Kurzbeschreibung": "Anliegen zu Bäumen, Gegenständen und abgestellten Fahrrädern.",
        "Beispielhafte Hinweise": "Baum auf dem Weg, Warnbarke unter einem Baum, Fahrrad ohne Sattel",
        "Erwartete Begriffe": ("baum", "quer", "umgestürzt"),
    },
    10: {
        "Thema": "Abgestellte Fahrzeuge",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Abgestellte Roller und andere Fahrzeuge, teilweise ohne Kennzeichen.",
        "Beispielhafte Hinweise": "Roller längere Zeit abgestellt, Fahrzeug ohne Kennzeichen",
        "Erwartete Begriffe": ("roller", "versicherungsschild", "gueltiges"),
    },
    11: {
        "Thema": "Gemischte Meldungen (Topic 11)",
        "Einordnung": "Gemischtes Topic",
        "Kurzbeschreibung": "Gegenstände im Wasser, Verschmutzungen und weitere unterschiedliche Anliegen.",
        "Beispielhafte Hinweise": "Abfall im Wasser, rutschige Treppe, abgestelltes Fahrzeug",
        "Erwartete Begriffe": ("wasser", "verstopft", "läuft"),
    },
    12: {
        "Thema": "Gemischte Meldungen (Topic 12)",
        "Einordnung": "Gemischtes Topic",
        "Kurzbeschreibung": "Meldungen zu Spielplätzen, Abfallbehältern, Müll und Wegschäden.",
        "Beispielhafte Hinweise": "Spielzeug defekt, Müllsäcke abgestellt, Schlaglöcher am Spielplatz",
        "Erwartete Begriffe": ("spielplatz", "vermuellt", "mülleimer"),
    },
}


def build_decision_overview(model, vocabulary, assignments, documents, summary, descriptions):
    """Beschriftungen mit den unveränderten Häufigkeiten der Modellzuordnung verbinden."""
    required = {"Topic", "Anzahl Meldungen", "Anteil Prozent", "Bezugsmenge Meldungen"}
    if not required.issubset(summary.columns):
        raise ValueError("In der technischen Themenübersicht fehlen benötigte Spalten.")
    if len(documents) == 0 or not documents.index.is_unique:
        raise ValueError("Die Meldungen müssen einen nicht leeren, eindeutigen Index haben.")
    if not assignments.index.equals(documents.index):
        raise ValueError("Meldungen und Topic-Zuordnungen passen nicht zusammen.")
    if model.n_components != 12 or set(descriptions) != set(range(1, 13)):
        raise ValueError("Diese Beschriftungen gehören zum ausgewählten Modell mit zwölf Topics.")
    if summary["Topic"].duplicated().any() or set(summary["Topic"]) != set(descriptions):
        raise ValueError("Die technische Übersicht muss jedes Topic genau einmal enthalten.")
    if not assignments["Topic"].dropna().isin(descriptions).all():
        raise ValueError("Mindestens eine Meldung hat eine unbekannte Topic-Nummer.")
    vocabulary = np.asarray(vocabulary)
    components = np.asarray(model.components_)
    if components.shape != (12, len(vocabulary)) or not np.isfinite(components).all():
        raise ValueError("Topic-Vektoren und Vokabular passen nicht zusammen.")

    # Die führenden Begriffe dienen als Kontrolle gegen offensichtlich vertauschte Topics.
    # Bei Änderungen an den Daten oder dem Modell müssen die Beschriftungen erneut geprüft werden.
    for topic, description in descriptions.items():
        leading = vocabulary[np.argsort(components[topic - 1])[::-1][:3]]
        if set(leading) != set(description["Erwartete Begriffe"]):
            raise ValueError(f"Topic {topic} hat andere führende Begriffe. Beschriftung prüfen.")

    counts = assignments["Topic"].value_counts().reindex(range(1, 13), fill_value=0)
    numeric = summary[list(required)].copy().set_index("Topic").sort_index()
    if not np.array_equal(numeric["Anzahl Meldungen"].to_numpy(), counts.to_numpy()):
        raise ValueError("Die Häufigkeiten weichen von den aktuellen Zuordnungen ab.")
    denominator = len(documents)
    shares = (100 * counts / denominator).round(1)
    if not (numeric["Bezugsmenge Meldungen"] == denominator).all():
        raise ValueError("Die Prozentanteile haben eine abweichende Bezugsgröße.")
    if not np.allclose(numeric["Anteil Prozent"], shares, rtol=0, atol=1e-10):
        raise ValueError("Die Prozentanteile passen nicht zu den Häufigkeiten.")

    meta = pd.DataFrame.from_dict(descriptions, orient="index").rename_axis("Topic").reset_index()
    result = numeric.reset_index().merge(
        meta.drop(columns="Erwartete Begriffe"), on="Topic", how="left", validate="one_to_one"
    )
    text_columns = ["Thema", "Einordnung", "Kurzbeschreibung", "Beispielhafte Hinweise"]
    if result[text_columns].isna().any().any():
        raise ValueError("Mindestens eine Beschriftung fehlt.")
    result = result.sort_values(["Anzahl Meldungen", "Topic"], ascending=[False, True]).reset_index(drop=True)
    result.insert(0, "Rang", np.arange(1, len(result) + 1))
    return result


_required_for_labels = (
    "SELECTED_TOPIC_COUNTS", "TOPIC_LABELS_REVIEWED", "nmf_model", "nmf_assignments",
    "df_model", "nmf_summary", "tfidf_vectorizer",
)
_missing_for_labels = [name for name in _required_for_labels if name not in globals()]
if _missing_for_labels:
    raise RuntimeError("Zuerst die Schritte bis 14 ausführen. Es fehlen: " + ", ".join(_missing_for_labels))
if SELECTED_TOPIC_COUNTS.get("NMF") != 12:
    raise ValueError("Die Beschriftungen sind für NMF mit zwölf Topics vorgesehen.")
if not TOPIC_LABELS_REVIEWED:
    raise RuntimeError("Zuerst die neuen Beschriftungen einsetzen und TOPIC_LABELS_REVIEWED aktivieren.")

decision_table = build_decision_overview(
    nmf_model, tfidf_vectorizer.get_feature_names_out(), nmf_assignments,
    df_model, nmf_summary, topic_descriptions,
)
print("Beschriftete Topics:", len(decision_table))
print("Davon als gemischt gekennzeichnet:", int(decision_table["Einordnung"].eq("Gemischtes Topic").sum()))



In [ ]:
# Schritt 16 Tabelle und Übersicht für Entscheidungstragende trennen

import numpy as np
import pandas as pd


def prepare_audience_overview(table):
    """Benannte Schwerpunkte anzeigen und die übrigen Meldungen getrennt ausweisen."""
    columns = [
        "Rang", "Thema", "Anzahl Meldungen", "Anteil Prozent", "Einordnung",
        "Kurzbeschreibung", "Beispielhafte Hinweise", "Topic", "Bezugsmenge Meldungen",
    ]
    if not isinstance(table, pd.DataFrame) or not set(columns).issubset(table.columns):
        raise ValueError("Die beschriftete Tabelle aus Schritt 15 fehlt oder ist unvollständig.")
    if table.empty:
        raise ValueError("Die beschriftete Tabelle ist leer.")
    full = table[columns].copy()
    if full[columns].isna().any().any():
        raise ValueError("Die beschriftete Tabelle enthält fehlende Angaben.")
    if full["Topic"].duplicated().any():
        raise ValueError("Eine Topic-Nummer kommt mehrfach vor.")
    allowed = {"Thematischer Schwerpunkt", "Gemischtes Topic"}
    if not full["Einordnung"].isin(allowed).all():
        raise ValueError("Die Tabelle enthält eine unbekannte Einordnung.")

    counts = pd.to_numeric(full["Anzahl Meldungen"], errors="raise").to_numpy(dtype=float)
    bases = pd.to_numeric(full["Bezugsmenge Meldungen"], errors="raise").to_numpy(dtype=float)
    shares = pd.to_numeric(full["Anteil Prozent"], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(counts).all() or (counts < 0).any() or (counts != np.floor(counts)).any():
        raise ValueError("Die Häufigkeiten müssen nicht negative ganze Zahlen sein.")
    if not np.isfinite(bases).all() or (bases <= 0).any() or (bases != np.floor(bases)).any():
        raise ValueError("Die Bezugsgröße muss eine positive ganze Zahl sein.")
    if not (bases == bases[0]).all():
        raise ValueError("Die Prozentwerte beziehen sich auf unterschiedliche Gesamtzahlen.")
    total = int(bases[0])
    assigned = int(counts.sum())
    if assigned > total:
        raise ValueError("Die Summe der Zuordnungen ist größer als die Bezugsgröße.")
    if not np.allclose(shares, np.round(100 * counts / total, 1), rtol=0, atol=1e-10):
        raise ValueError("Die Prozentwerte passen nicht zu den Häufigkeiten.")

    # Alle zwölf Topics bleiben in der vollständigen Tabelle erhalten.
    full = full.sort_values(["Anzahl Meldungen", "Topic"], ascending=[False, True]).reset_index(drop=True)
    full["Rang"] = np.arange(1, len(full) + 1)
    shown = full.loc[full["Einordnung"].eq("Thematischer Schwerpunkt")].copy()
    if shown.empty:
        raise ValueError("Es sind keine benannten Schwerpunkte für die Übersicht vorhanden.")
    shown = shown.reset_index(drop=True)
    shown["Rang"] = np.arange(1, len(shown) + 1)
    n_shown = int(shown["Anzahl Meldungen"].sum())
    n_mixed = int(full.loc[full["Einordnung"].eq("Gemischtes Topic"), "Anzahl Meldungen"].sum())
    n_unassigned = total - assigned
    if n_shown + n_mixed + n_unassigned != total:
        raise RuntimeError("Die Teilmengen ergeben nicht die Gesamtzahl der Meldungen.")

    # Diese Zusammenfassung beschreibt die Darstellung und nicht die Genauigkeit der Erkennung.
    coverage = pd.DataFrame({
        "Darstellung": ["In der Themenübersicht", "Nicht nach Themen aufgeschlüsselt"],
        "Anzahl Meldungen": [n_shown, n_mixed + n_unassigned],
        "Anteil Prozent": [round(100 * n_shown / total, 1), round(100 * (n_mixed + n_unassigned) / total, 1)],
        "Bezugsmenge Meldungen": [total, total],
    })
    public_columns = [
        "Rang", "Thema", "Anzahl Meldungen", "Anteil Prozent",
        "Kurzbeschreibung", "Beispielhafte Hinweise", "Bezugsmenge Meldungen",
    ]
    return full, shown[public_columns].copy(), coverage


if "decision_table" not in globals() or decision_table is None:
    raise RuntimeError("Zuerst Schritt 15 mit den aktuellen Beschriftungen ausführen.")

decision_table_all, decision_table_final, decision_coverage = prepare_audience_overview(decision_table)

decision_display = decision_table_final[["Rang", "Thema", "Anzahl Meldungen", "Anteil Prozent"]].copy()
decision_display["Anteil Prozent"] = decision_display["Anteil Prozent"].map(
    lambda value: f"{value:.1f} %".replace(".", ",")
)
display(decision_display.rename(columns={"Anteil Prozent": "Anteil"}))
print("Meldungen in dargestellten Schwerpunkten:", int(decision_coverage.iloc[0]["Anzahl Meldungen"]))
print("Nicht nach Themen aufgeschlüsselt:", int(decision_coverage.iloc[1]["Anzahl Meldungen"]))
print("Bezugsgröße der Prozentanteile:", int(decision_coverage.iloc[0]["Bezugsmenge Meldungen"]))
print("Technische Kontrolle der Darstellung: bestanden")


In [ ]:
# Schritt 17 Die Tabelle und die kommunale Übersicht speichern

from pathlib import Path

required_tables = ("decision_table_all", "decision_table_final", "decision_coverage", "OUTPUT_DIR")
missing_tables = [name for name in required_tables if name not in globals()]
if missing_tables:
    raise RuntimeError("Zuerst Schritt 16 ausführen. Es fehlen: " + ", ".join(missing_tables))

_decision_output = Path(OUTPUT_DIR)
_decision_output.mkdir(parents=True, exist_ok=True)

# Die vollständige Tabelle enthält auch die gemischten Topics und ihre Kennungen.
_export_tables = {
    "nmf_topics_beschriftet.csv": decision_table_all,
    "entscheidungstraeger_themenuebersicht.csv": decision_table_final,
    "umfang_themenuebersicht.csv": decision_coverage,
}
for _filename, _table in _export_tables.items():
    _path = _decision_output / _filename
    _table.to_csv(_path, index=False, sep=";", encoding="utf-8-sig")
    print("Gespeichert:", _path.resolve())


In [ ]:
# Schritt 18 Die Schwerpunkte als Balkendiagramm darstellen

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

_required_chart = ("decision_table_final", "decision_coverage", "OUTPUT_DIR")
_missing_chart = [name for name in _required_chart if name not in globals()]
if _missing_chart:
    raise RuntimeError("Zuerst Schritt 16 ausführen. Es fehlen: " + ", ".join(_missing_chart))

chart_df = decision_table_final.sort_values("Rang").copy()
if chart_df.empty or len(decision_coverage) != 2:
    raise ValueError("Für das Diagramm fehlen die Schwerpunkte oder die Gesamtzahlen.")
denominator = int(decision_coverage.iloc[0]["Bezugsmenge Meldungen"])
shown_count = int(chart_df["Anzahl Meldungen"].sum())
other_count = int(decision_coverage.iloc[1]["Anzahl Meldungen"])
if shown_count != int(decision_coverage.iloc[0]["Anzahl Meldungen"]) or shown_count + other_count != denominator:
    raise ValueError("Diagramm und Zusammenfassung beziehen sich auf unterschiedliche Ergebnisse.")
if not np.allclose(chart_df["Anteil Prozent"], (100 * chart_df["Anzahl Meldungen"] / denominator).round(1)):
    raise ValueError("Die Prozentwerte des Diagramms haben eine falsche Bezugsgröße.")

def format_number(value):
    return f"{int(value):,}".replace(",", ".")

def format_share(value):
    return f"{value:.1f} %".replace(".", ",")

fig, ax = plt.subplots(figsize=(12, 6.8))
positions = np.arange(len(chart_df))
bars = ax.barh(positions, chart_df["Anzahl Meldungen"])
ax.set_yticks(positions, labels=chart_df["Thema"])
ax.invert_yaxis()
ax.set_xlabel("Anzahl der automatisch zugeordneten Meldungen")
ax.set_title("Schwerpunkte der Meldungen", loc="left", pad=30, fontsize=15)
ax.text(0, 1.035, "Automatische Themenanalyse · Mängelmelder Konstanz", transform=ax.transAxes, fontsize=10)
ax.set_xlim(0, max(1, float(chart_df["Anzahl Meldungen"].max())) * 1.25)
ax.spines[["top", "right"]].set_visible(False)
ax.bar_label(
    bars,
    labels=[f"{format_number(count)} ({format_share(share)})" for count, share in zip(chart_df["Anzahl Meldungen"], chart_df["Anteil Prozent"])],
    padding=5, fontsize=10,
)

# Die nicht dargestellten Meldungen bleiben Teil der Bezugsgröße.
footer = (
    f"Dargestellt: {format_number(shown_count)} Meldungen ({format_share(100 * shown_count / denominator)}).\n"
    f"Nicht nach Themen aufgeschlüsselt: {format_number(other_count)} Meldungen ({format_share(100 * other_count / denominator)}).\n"
    f"Basis: {format_number(denominator)} Meldungen nach Bereinigung. Keine Rangfolge nach Dringlichkeit."
)
fig.text(0.035, 0.025, footer, fontsize=9.5, va="bottom", linespacing=1.6)
fig.subplots_adjust(left=0.30, right=0.98, top=0.83, bottom=0.24)
chart_path = Path(OUTPUT_DIR) / "themenhaeufigkeiten_nmf.png"
chart_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Diagramm gespeichert:", chart_path.resolve())